# Notebook 2 — Data Inspection
### Sprint 4 | Data Inspection & Exploratory Data Analysis (EDA)

**Methodology for every topic below:**
1. **Understand the Concept** (Markdown) — explained in my own words.
2. **Demonstrate the Concept** (Markdown) — a simple / real-world / business / AI-ML
   example, and why it matters during data analysis.
3. **Implement the Concept** (Python) — using Pandas/NumPy, with what the code does, why
   the method was used, what the output means, what insights it gives, and how it can
   affect an ML pipeline.

**Dataset:** Telco Customer Churn (7,043 customers, 21 columns) — same dataset documented
in `01_Dataset_Understanding.ipynb`. This notebook is the systematic **Inspect** step of
this sprint's Load → Inspect → Understand → Analyze → Visualize → Identify
Problems → Document Findings principle.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Dataset loaded: 7,043 rows, 21 columns


---
## 1. Number of Rows

### Step 1 — Understand the Concept
The number of rows tells me how many individual observations (here, customers) the
dataset contains — the foundation for judging whether there's enough data for reliable
analysis or model training.

### Step 2 — Demonstrate the Concept
**Business example:** Knowing there are 7,043 customer records tells a business analyst
whether this is a small regional sample or a large enough dataset to represent the whole
customer base meaningfully.

**Why it matters during data analysis:** Row count is the first number I check — too few
rows and even a well-designed analysis may not generalize; enough rows and I can trust
patterns found are less likely to be pure chance (Sprint 2, Notebook 7).

### Step 3 — Implement the Concept


In [2]:
num_rows = len(df)
print(f"Number of rows: {num_rows:,}")


Number of rows: 7,043


**What this does:** Uses Python's built-in `len()` on the DataFrame, which returns
its row count. **Why this method:** `len(df)` is the simplest, most direct way to get row
count alone (versus `.shape`, which gives both dimensions at once — covered later).
**What the output means:** 7,043 individual customer records. **Insight:** a solid sample
size for detecting genuine churn patterns rather than noise. **ML pipeline impact:** row
count directly determines what train/test split sizes and cross-validation strategies are
sensible later.


---
## 2. Number of Columns

### Step 1 — Understand the Concept
The number of columns tells me how many distinct pieces of information were recorded
about each observation — how "wide" the dataset is.

### Step 2 — Demonstrate the Concept
**Business example:** 21 columns means each customer record captures demographics,
services, contract details, and billing — a fairly rich profile, but still manageable to
review column-by-column.

**Why it matters during data analysis:** Column count sets the scope of the inspection
work ahead — a 21-column dataset can be reviewed thoroughly one column at a time; a
500-column dataset would need a more automated, summary-driven approach.

### Step 3 — Implement the Concept


In [3]:
num_columns = len(df.columns)
print(f"Number of columns: {num_columns}")


Number of columns: 21


**What this does:** Counts the entries in `df.columns`. **Why this method:**
directly counting the columns index is the most explicit way to isolate just this one
dimension. **What the output means:** 21 distinct recorded attributes per customer.
**Insight:** a manageable number of columns to inspect individually, which is exactly
the approach this notebook takes. **ML pipeline impact:** with a moderate column count,
manually reviewing each column's meaning and quality (rather than only automated checks)
is realistic and worthwhile.


---
## 3. Data Types

### Step 1 — Understand the Concept
(Recap from Sprint 3) Every column has a data type (`dtype`) — Pandas' internal record of
whether it's storing numbers, text, or another kind of value — which determines what
operations are valid on that column.

### Step 2 — Demonstrate the Concept
**Business example:** `MonthlyCharges` should be numeric (so averages and comparisons
work); `Contract` should be text/categorical (so it can be grouped and counted).

**Why it matters during data analysis:** A column's *stored* dtype doesn't always match
its *true* nature — as flagged in Notebook 1, `TotalCharges` should be numeric but is
currently stored as text, which is exactly the kind of issue this check exists to catch.

### Step 3 — Implement the Concept


In [4]:
print(df.dtypes)


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


**What this does:** Prints every column's current stored data type. **Why this
method:** `.dtypes` gives a complete, one-line-per-column technical summary in a single
call. **What the output means:** most text fields show `object`, numeric fields show
`int64`/`float64` — but `TotalCharges` shows `object` despite representing money.
**Insight:** confirms the data-quality issue flagged in Notebook 1 — `TotalCharges` needs
explicit conversion to numeric. **ML pipeline impact:** any column with a mismatched
dtype will either cause errors or be silently mishandled (e.g., treated as categorical)
if not corrected before modeling.


---
## 4. Column Names

### Step 1 — Understand the Concept
Column names are the labels used to reference each variable in code — checking them
confirms exactly what's available and catches naming issues (typos, inconsistent casing,
extra spaces) before they cause errors later.

### Step 2 — Demonstrate the Concept
**Business example:** Confirming the exact spelling `SeniorCitizen` (not `Senior_Citizen`
or `senior_citizen`) before referencing it anywhere else in the analysis.

**Why it matters during data analysis:** Referencing a mistyped or incorrectly-cased
column name is one of the most common early coding errors — checking `.columns` first
avoids this entirely.

### Step 3 — Implement the Concept


In [5]:
print(list(df.columns))


['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


**What this does:** Lists every column name exactly as stored. **Why this method:**
`.columns` (converted to a list for cleaner printing) gives the authoritative, exact
naming to reference going forward. **What the output means:** 21 column names, using
mixed casing conventions (e.g., `gender` is lowercase, `SeniorCitizen` is CamelCase) —
worth noting as a minor inconsistency. **Insight:** column names are otherwise clear and
descriptive, requiring no immediate renaming for this analysis. **ML pipeline impact:**
inconsistent naming conventions are a common candidate for cleanup (Sprint 3, Notebook 8)
if this dataset were to be merged with another source using a different convention.


---
## 5. Index

### Step 1 — Understand the Concept
(Recap from Sprint 3) The index is the row label of the DataFrame — by default, a simple
0, 1, 2, ... sequence, though it can be replaced with something more meaningful.

### Step 2 — Demonstrate the Concept
**Business example:** Right now, rows are labeled 0 to 7,042 by default — but since
`customerID` is a genuine, unique identifier, it could be set as the index for more
meaningful row lookups.

**Why it matters during data analysis:** Checking the index confirms whether row lookups
will be by meaningless position or by a meaningful business key.

### Step 3 — Implement the Concept


In [6]:
print("Current index:", df.index)

# Demonstrating a more meaningful index, without permanently changing df
indexed_by_id = df.set_index('customerID')
print("\nAfter setting customerID as the index:")
print(indexed_by_id.index[:5])


Current index: RangeIndex(start=0, stop=7043, step=1)

After setting customerID as the index:
Index(['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU'], dtype='str', name='customerID')


**What this does:** Prints the default numeric index, then demonstrates setting
`customerID` as a more meaningful index instead. **Why this method:** `.set_index()`
directly swaps in a meaningful business key for row lookups. **What the output means:** by
default, rows are labeled 0-7,042; after `.set_index()`, each row is labeled by its actual
customer ID. **Insight:** since `customerID` is confirmed unique (Notebook 1), it's a
valid, safe choice for an index if direct customer lookups become useful later. **ML
pipeline impact:** keeping the default numeric index is usually fine for model training
itself, but a meaningful index helps when tracing specific predictions back to specific
customers during evaluation.


---
## 6. `head()`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.head()` previews the first few rows of a DataFrame — the fastest
first look after loading any dataset.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Immediately after `pd.read_csv()`, `.head()` is almost always the
very next command run, confirming the file loaded correctly and columns look as expected.

### Step 3 — Implement the Concept


In [7]:
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


**What this does:** Shows the first 5 customer records with all 21 columns.
**Why this method:** it's the fastest sanity check that the CSV loaded correctly, with
proper column alignment. **What the output means:** real customer data, sensibly
formatted — no obvious loading errors (misaligned columns, garbled text). **Insight:**
data looks well-formed at a glance, though `TotalCharges` values are visible as plain
numbers that are secretly stored as text. **ML pipeline impact:** confirming a clean load
here prevents chasing phantom bugs later that actually stem from a bad file read.


---
## 7. `tail()`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.tail()` previews the last few rows — confirming the dataset wasn't
truncated or cut off during loading.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Checking `.tail()` after loading a large file downloaded from a URL
confirms the full file actually arrived, rather than being cut short by a network issue.

### Step 3 — Implement the Concept


In [8]:
df.tail()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


**What this does:** Shows the last 5 customer records. **Why this method:** it
verifies the data extends all the way to the expected final row, not stopping early.
**What the output means:** the last rows look just as well-formed as the first —
confirming a complete, untruncated load of all 7,043 rows. **Insight:** no evidence of a
partial or corrupted download. **ML pipeline impact:** confirming complete data loading
is especially important when working with data fetched from an external source (a
network issue silently truncating a file is a real, if uncommon, risk).


---
## 8. `sample()`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.sample()` returns a random selection of rows — a less biased
preview than always looking at the same first or last rows.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** If the CSV happened to be sorted or grouped by some hidden order (e.g.,
by signup date, or by region), `.head()` alone could give a misleading first impression —
a random sample avoids that risk.

### Step 3 — Implement the Concept


In [9]:
df.sample(5, random_state=42)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,...,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


**What this does:** Randomly selects 5 rows from anywhere in the dataset, using a
fixed seed for reproducibility (Sprint 3, Notebook 3). **Why this method:** a random
sample avoids any bias that might come from the data's original ordering. **What the
output means:** a genuinely representative mini cross-section of the dataset. **Insight:**
values across this random sample look consistent with the `.head()`/`.tail()` previews —
no obvious hidden ordering issue. **ML pipeline impact:** this same random-sampling
principle underlies how train/test splits are created (Sprint 2, Notebook 7) — a biased
split would give a misleadingly optimistic evaluation.


---
## 9. `shape`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.shape` reports both dimensions at once, as
`(number_of_rows, number_of_columns)` — the single most efficient way to check dataset
size.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Checking `.shape` before and after any filtering or cleaning step
confirms exactly how many rows (or columns) were affected by that step.

### Step 3 — Implement the Concept


In [10]:
print(f"Shape: {df.shape}")


Shape: (7043, 21)


**What this does:** Reports `(7043, 21)` in a single call. **Why this method:**
combines what Topics 1 and 2 checked separately into one efficient call — the standard,
preferred way to check size in practice. **What the output means:** confirms, once more,
7,043 rows and 21 columns. **Insight:** consistent with everything checked so far — no
surprises. **ML pipeline impact:** `.shape` is the single most-repeated sanity check
throughout a real data pipeline, run after nearly every transformation step.


---
## 10. `info()`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.info()` gives a compact technical summary — column names,
non-null counts, dtypes, and memory usage — all in one call.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** `.info()` is typically the single most-used first command for
spotting missing values and dtype issues at a glance, before any deeper column-by-column
work.

### Step 3 — Implement the Concept


In [11]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**What this does:** Prints every column's name, non-null count, and dtype, plus
total memory usage. **Why this method:** it's the fastest way to get a full technical
overview in one call. **What the output means:** every column shows 7,043 non-null values
— meaning **no column has an official `NaN`/missing value** by Pandas' standard detection.
**Insight:** this is a genuinely important, slightly deceptive finding — `.info()` reports
zero missing values, yet we already know `TotalCharges` has 11 blank-string entries. This
shows *why* `.info()` alone isn't enough: a blank string (`""`) is not the same as a true
`NaN`, so Pandas doesn't flag it here. **ML pipeline impact:** relying on `.info()` alone
would miss this real data-quality issue entirely — a deeper, column-specific check
(done next, and in Notebook 3) is required to catch it.


---
## 11. `describe()`

### Step 1 — Understand the Concept
(Recap from Sprint 2/3) `.describe()` generates summary statistics (count, mean, std,
quartiles) for every numeric column at once.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** `.describe()` is the fastest way to spot implausible values — a
negative age, a percentage above 100, or (as will be relevant here) a suspiciously narrow
set of "numeric" columns.

### Step 3 — Implement the Concept


In [12]:
print(df.describe())

print("\nDescribing ALL columns, including categorical ones:")
print(df.describe(include='all').T[['count', 'unique', 'top', 'freq']])


       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000

Describing ALL columns, including categorical ones:
                   count unique               top  freq
customerID          7043   7043        7590-VHVEG     1
gender              7043      2              Male  3555
SeniorCitizen     7043.0    NaN               NaN   NaN
Partner             7043      2                No  3641
Dependents          7043      2                No  4933
tenure            7043.0    NaN               NaN   NaN
PhoneService        7043      2               Yes  6361
MultipleLines       7043      3                No

**What this does:** Computes statistics for numeric columns, then separately
summarizes every column (numeric and categorical) with `include='all'`. **Why this
method:** the default `.describe()` only covers 3 columns (`SeniorCitizen`, `tenure`,
`MonthlyCharges`) — a strong, visible confirmation that `TotalCharges` is NOT being
treated as numeric. **What the output means:** `tenure` ranges 0-72 months,
`MonthlyCharges` ranges roughly $18-$119, both plausible; the `include='all'` view shows
`TotalCharges`'s `top`/`freq` output (categorical-style), confirming it's being treated
as text. **Insight:** this is the clearest evidence yet that `TotalCharges` needs to be
converted to a proper numeric type before any real analysis. **ML pipeline impact:**
without fixing this, `TotalCharges` — a potentially important churn predictor — would be
entirely unusable as a numeric feature.


---
## 12. `dtypes`

### Step 1 — Understand the Concept
Revisiting `dtypes` specifically to formally list which columns need a type correction
before moving forward, based on everything found so far.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Explicitly listing "columns needing type correction" is a standard
documented output of the inspection phase, handed off directly into the cleaning phase of
a real pipeline.

### Step 3 — Implement the Concept


In [13]:
print("Current dtypes:")
print(df.dtypes.value_counts())

print("\nColumns flagged for type correction:")
print(" - TotalCharges: stored as 'object' (text), should be numeric (float)")


Current dtypes:
str        18
int64       2
float64     1
Name: count, dtype: int64

Columns flagged for type correction:
 - TotalCharges: stored as 'object' (text), should be numeric (float)


**What this does:** Summarizes how many columns fall into each dtype category, then
explicitly documents the one column needing correction. **Why this method:**
`.value_counts()` on `.dtypes` gives a quick "how many of each type" overview.
**What the output means:** 16 `object` columns, 1 `int64`, 2 `float64`/`int` numeric —
with one of those 16 `object` columns (`TotalCharges`) actually being miscategorized.
**Insight:** formally documenting this now creates a clear, actionable finding for the
cleaning stage of this sprint. **ML pipeline impact:** this kind of explicit "flagged
columns" list is exactly what a real data-quality report would hand off to whoever
performs data cleaning next.


---
## 13. `nunique()`

### Step 1 — Understand the Concept
(Recap from Sprint 3) `.nunique()` counts the distinct values in each column — essential
for telling apart identifiers, low-cardinality categoricals, and genuinely continuous
numeric columns.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** A column with `nunique() == len(df)` is almost certainly an
identifier; a column with very few unique values relative to row count is almost
certainly categorical — this single check helps classify every column at a glance.

### Step 3 — Implement the Concept


In [14]:
print(df.nunique().sort_values())


gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
PhoneService           2
PaperlessBilling       2
Churn                  2
MultipleLines          3
TechSupport            3
StreamingTV            3
OnlineBackup           3
DeviceProtection       3
StreamingMovies        3
Contract               3
OnlineSecurity         3
InternetService        3
PaymentMethod          4
tenure                73
MonthlyCharges      1585
TotalCharges        6531
customerID          7043
dtype: int64

**What this does:** Counts distinct values per column, sorted from fewest to most.
**Why this method:** sorting makes the pattern immediately visible — binary/small
categorical columns cluster at the top, `MonthlyCharges` and `TotalCharges` (high
cardinality) at the bottom, and `customerID` (matching row count exactly) at the very
end. **What the output means:** most service-related columns have only 2-3 unique values
(confirming they're categorical), while `customerID` has 7,043 (confirming it's a pure
identifier). **Insight:** this single check nicely corroborates the numerical/categorical/
identifier classification built up across this notebook and Notebook 1. **ML pipeline
impact:** `.nunique()` is often the fastest automated way to auto-classify columns before
deciding an encoding strategy for each one.


---
## 14. `value_counts()`

### Step 1 — Understand the Concept
(Recap from Sprint 2/3) `.value_counts()` shows exactly how often each distinct value
occurs in a column — going one level deeper than `.nunique()`'s simple count of how many
distinct values exist.

### Step 2 — Demonstrate the Concept
**AI/ML use case:** Checking `.value_counts()` on every categorical column (and the
target) is standard practice for spotting class imbalance, unexpected category labels, or
inconsistent category spellings (Sprint 3, Notebook 12).

### Step 3 — Implement the Concept


In [15]:
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()


--- Contract ---
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

--- PaymentMethod ---


PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64



**What this does:** Prints the exact frequency breakdown for three representative
categorical columns. **Why this method:** spot-checking a few key categorical columns
this way catches unexpected labels (e.g., a stray misspelled category) before they cause
problems downstream. **What the output means:** `Contract` is dominated by month-to-month
customers; `InternetService` splits across DSL, Fiber optic, and "No"; `PaymentMethod` has
4 clean, sensible categories — no inconsistent spellings or surprise values found in any
of these three columns. **Insight:** these category distributions themselves are useful
business information — e.g., month-to-month customers being the largest group is worth
investigating against the churn rate in the next (Analyze) stage of this sprint. **ML
pipeline impact:** clean, well-understood categories like these are ready for one-hot
encoding without needing category consolidation first.


---
## Column Classification Summary

Based on everything inspected in this notebook, every column is classified below —
exactly the deliverable this notebook's brief calls for.


In [16]:
column_classification = {
    'Numerical columns (currently correct)': ['SeniorCitizen', 'tenure', 'MonthlyCharges'],
    'Numerical column needing correction': ['TotalCharges (stored as text — must convert to float)'],
    'Categorical columns': [c for c in df.select_dtypes(include='object').columns
                             if c not in ['customerID', 'TotalCharges']],
    'Date/time columns': ['None present in this dataset'],
    'Identifier columns': ['customerID'],
    'Potential target column': ['Churn (binary classification target: Yes/No)'],
}

for category, cols in column_classification.items():
    print(f"{category}:")
    for c in cols:
        print(f"   - {c}")
    print()


Numerical columns (currently correct):
   - SeniorCitizen
   - tenure
   - MonthlyCharges

Numerical column needing correction:
   - TotalCharges (stored as text — must convert to float)

Categorical columns:
   - gender
   - Partner
   - Dependents
   - PhoneService
   - MultipleLines
   - InternetService
   - OnlineSecurity
   - OnlineBackup
   - DeviceProtection
   - TechSupport
   - StreamingTV
   - StreamingMovies
   - Contract
   - PaperlessBilling
   - PaymentMethod
   - Churn

Date/time columns:
   - None present in this dataset

Identifier columns:
   - customerID

Potential target column:
   - Churn (binary classification target: Yes/No)



/tmp/ipykernel_559/2490922580.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  'Categorical columns': [c for c in df.select_dtypes(include='object').columns


**Insight:** This dataset has no date/time columns at all — every customer record
is a single snapshot rather than a time series, which rules out any time-based feature
engineering (Sprint 3, Notebook 11) for this particular dataset. **ML pipeline impact:**
this classification is the direct hand-off point into data cleaning and feature
engineering — every category above maps to a different preparation step (numeric columns
get scaled, categorical columns get encoded, the identifier gets dropped, the target gets
label-encoded).


---
## Summary

| Concept | Finding for THIS dataset |
|---|---|
| Number of Rows | 7,043 |
| Number of Columns | 21 |
| Data Types | 16 object, 1 int64, 2 numeric (1 miscategorized: `TotalCharges`) |
| Column Names | Clear and descriptive; minor casing inconsistency noted |
| Index | Default 0-7,042; `customerID` available as a meaningful alternative |
| head() / tail() / sample() | Confirms a clean, complete, untruncated load |
| shape | (7043, 21) |
| info() | Reports 0 missing values — **misleading**, since blanks in `TotalCharges` aren't true NaN |
| describe() | Confirms only 3 columns are truly treated as numeric right now |
| dtypes | 1 column (`TotalCharges`) flagged for type correction |
| nunique() | Confirms categorical vs. identifier classification |
| value_counts() | No inconsistent category spellings found in spot-checked columns |

**Key finding carried forward:** `.info()` reporting "no missing values" does **not** mean
the data is actually complete — `TotalCharges` has 11 blank-string entries that Pandas
doesn't recognize as `NaN` under its default text dtype. This is the central problem the
next notebooks in this sprint (data quality / missing values) will need to properly
investigate and resolve.

**Next notebook:** topics to be confirmed for Sprint 4, Notebook 3.
